# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [21]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


SparkSession creada
Versión de Spark: 4.1.1


In [22]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [23]:
# Lectura del dataset de Clientes
df_clientes = (spark.read
    .option("sep", ";")           # Especificamos el punto y coma como separador de columnas
    .option("header", True)       # Indicamos que la primera fila contiene los nombres de las columnas
    .option("inferSchema", True)  # Spark escanea los datos para detectar los tipos (entero, texto, etc.)
    .csv(str(clientes_path)))     # Convertimos el objeto Path a string para que PySpark lo lea sin fallos

# Verificamos que la carga es correcta mostrando el esquema y las primeras filas
df_clientes.printSchema()
df_clientes.show(15)

# Lectura del dataset de Pedidos (aplicamos exactamente la misma configuración)
df_pedidos = (spark.read
    .option("sep", ";")
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(pedidos_path)))

# Verificamos la carga de pedidos
df_pedidos.printSchema()
df_pedidos.show(15)

root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)

+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
|         6|Carlos| Alicante |Estandar|
|         7| Elena|    Bilbao|Estandar|
|         8|Javier|    Madrid| Premium|
|         9|  Sara|    Murcia| Premium|
|        10| David|    Bilbao|Estandar|
|        11|Raquel|  Sevilla | Premium|
|        12|Miguel|  Zaragoza|Estandar|
|        13| Irene|    Madrid| Premium|
|        14|Sergio|   Murcia |Estandar|
|        15| Paula|   Granada|Estandar|
+----------+------+----------+--------+
only showing top 15 rows
root
 |-- id_pedido: integer (nullable = true)
 |-

[Stage 5:>                                                          (0 + 1) / 1]

+---------+----------+----------+-----------+--------+---------------+
|id_pedido|id_cliente|     fecha|   producto|cantidad|precio_unitario|
+---------+----------+----------+-----------+--------+---------------+
|     1001|        12|2025-03-05|      Ratón|     6.0|             16|
|     1002|        32|2025-02-02|      Ratón|     3.0|             19|
|     1003|         5|2025-03-07|    Monitor|     2.0|            210|
|     1004|        13|2025-03-18|  Impresora|     4.0|            205|
|     1005|        41|2025-03-10|    Teclado|     5.0|             40|
|     1006|         5|2025-03-05|    Teclado|    NULL|             35|
|     1007|        27|2025-02-13|      Ratón|     1.0|             23|
|     1008|         7|2025-02-04|     Webcam|     6.0|             77|
|     1009|        18|2025-03-02|    Teclado|     1.0|             38|
|     1010|        31|2025-02-14|     Webcam|     1.0|             78|
|     1011|        36|2025-03-15|Auriculares|     2.0|             32|
|     

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [24]:
# --- PERFILADO DE DATOS (DATA PROFILING) ---
# Antes de modificar nada, contamos cuántas filas tenemos en total
# para tener una referencia del tamaño original de nuestro dataset.
total_inicial = df_clientes.count()
print(f"Total de registros iniciales: {total_inicial}")

# Contamos cuántos duplicados exactos hay en origen. 
# La lógica es sencilla: restamos los registros únicos (dropDuplicates) al total de registros.
total_distintos = df_clientes.dropDuplicates().count()
print(f"Número de filas duplicadas antes de limpiar: {total_inicial - total_distintos}")

# Calculamos cuántos valores nulos hay en cada columna para conocer la "salud" de los datos.
# ¿Cómo funciona? isNull() da True/False, cast("int") lo pasa a 1/0, y sum() suma esos 1s.
print("Recuento de nulos por columna ANTES de limpiar:")
df_clientes.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_clientes.columns]
).show()
# --------------------------------------------------

# 1. Validar tipos de datos: 
# Forzamos que 'id_cliente' sea un número entero. Esto evita errores futuros 
# si inferSchema interpretó algún número con espacios raros como texto.
df_clientes = df_clientes.withColumn("id_cliente", F.col("id_cliente").cast("integer"))

# 2. Gestionar valores nulos: 
# - dropna: Eliminamos la fila completa si falta el 'id_cliente', ya que es nuestra clave principal para el join.
# - fillna: Si falta el nombre, la ciudad o el segmento, no borramos la fila, rellenamos el hueco con "Desconocido".
df_clientes = df_clientes.dropna(subset=["id_cliente"])
df_clientes = df_clientes.fillna("Desconocido", subset=["nombre", "ciudad", "segmento"])

# 3. Eliminar espacios en columnas de texto:
# F.trim limpia los espacios en blanco invisibles al principio y al final de los textos. 
# Así evitamos que Spark considere " Madrid" y "Madrid " como dos ciudades distintas.
df_clientes = (df_clientes
    .withColumn("nombre", F.trim(F.col("nombre")))
    .withColumn("ciudad", F.trim(F.col("ciudad")))
    .withColumn("segmento", F.trim(F.col("segmento")))
)

# 4. Eliminar duplicados:
# Aplicamos la eliminación real de los duplicados que detectamos arriba en el perfilado.
df_clientes = df_clientes.dropDuplicates()

# ==============================================================================
# Verificamos los resultados tras la limpieza
# Hacemos un nuevo count() para saber cuántos registros han sobrevivido.
total_final = df_clientes.count()
print(f"Total de registros DESPUÉS de limpiar: {total_final}")

# Mostramos el total exacto de filas eliminadas (por ser nulos críticos o duplicados)
print(f"Registros eliminados (nulos críticos o duplicados): {total_inicial - total_final}")

print("Datos limpios:")
df_clientes.show(15) # Mostramos 15 filas para tener una mejor visión general del resultado

Total de registros iniciales: 43
Número de filas duplicadas antes de limpiar: 3
Recuento de nulos por columna ANTES de limpiar:
+----------+------+------+--------+
|id_cliente|nombre|ciudad|segmento|
+----------+------+------+--------+
|         0|     0|     0|       0|
+----------+------+------+--------+

Total de registros DESPUÉS de limpiar: 40
Registros eliminados (nulos críticos o duplicados): 3
Datos limpios:
+----------+--------+--------+--------+
|id_cliente|  nombre|  ciudad|segmento|
+----------+--------+--------+--------+
|        18|   Jorge| Sevilla| Premium|
|        40|   Tomas|Valencia|Estandar|
|         5|   Lucia|  Bilbao| Premium|
|         8|  Javier|  Madrid| Premium|
|        17|    Alba|  Madrid|Estandar|
|         2|    Luis|  Bilbao| Premium|
|        33|Patricia|Alicante| Premium|
|        27|   Laura|  Bilbao|Estandar|
|        32|  Andres|Zaragoza| Premium|
|        21|  Noelia|  Malaga| Premium|
|        39|     Eva|Zaragoza|Estandar|
|        12|  Miguel

## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [25]:
# (Incluye limpieza completa: trim, duplicados y tratamiento de nulos)
# ==============================================================================

# --- PERFILADO DE DATOS (DATA PROFILING) ---
# Guardamos el total de filas original para compararlo al final del proceso.
total_inicial_pedidos = df_pedidos.count()
print(f"Total de registros iniciales en pedidos: {total_inicial_pedidos}")

# Calculamos los duplicados exactos restando los registros únicos al total.
# Si este número es muy alto en un entorno real, habría que avisar al equipo que genera el CSV.
total_distintos_pedidos = df_pedidos.dropDuplicates().count()
print(f"Número de filas duplicadas antes de limpiar: {total_inicial_pedidos - total_distintos_pedidos}")

# Radiografía de nulos: Vemos cuántos valores faltan en cada columna.
print("Recuento de nulos por columna ANTES de limpiar:")
df_pedidos.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_pedidos.columns]
).show()
# --------------------------------------------------

# 1. Validar y convertir tipos de datos:
# Aunque usamos 'inferSchema=True' al leer el CSV, siempre es mejor forzar los tipos manualmente
# para evitar que un carácter extraño convierta toda una columna numérica en texto.
df_pedidos = (df_pedidos
    .withColumn("id_pedido", F.col("id_pedido").cast("integer"))         # Forzamos a número entero
    .withColumn("id_cliente", F.col("id_cliente").cast("integer"))       # Clave para el join futuro
    .withColumn("fecha", F.to_date(F.col("fecha")))                      # Convierte texto a formato Date nativo de Spark
    .withColumn("cantidad", F.col("cantidad").cast("double"))            # Double permite decimales
    .withColumn("precio_unitario", F.col("precio_unitario").cast("double")) 
)

# 2. Gestionar valores nulos:
# - dropna: Si un pedido no tiene 'id_pedido' o no sabemos de qué cliente es ('id_cliente'),
#   ese registro es "basura" porque no podremos cruzarlo. Lo eliminamos por completo.
df_pedidos = df_pedidos.dropna(subset=["id_pedido", "id_cliente"])

# - fillna (textos): Si nos falta el nombre del producto, no borramos la venta, simplemente 
#   le ponemos la etiqueta "Desconocido" para no perder el importe económico de esa fila.
df_pedidos = df_pedidos.fillna("Desconocido", subset=["producto"])

# - fillna (números): Si falta la cantidad o el precio, ponemos un 0.0. 
#   ESTO ES CRÍTICO: Si intentamos multiplicar un número por un valor nulo (Null), Spark 
#   devolverá Null, y perderemos el cálculo. Con un 0.0, evitamos que el código falle.
df_pedidos = df_pedidos.fillna(0.0, subset=["cantidad", "precio_unitario"])

# 3. Eliminar espacios en columnas de texto (Trim):
# F.trim limpia espacios invisibles a los lados del texto.
# Evita que el sistema de análisis se confunda pensando que "Monitor" y " Monitor " son dos productos distintos.
df_pedidos = df_pedidos.withColumn("producto", F.trim(F.col("producto")))

# 4. Eliminar duplicados:
# Limpiamos las filas que sean 100% idénticas al resto.
df_pedidos = df_pedidos.dropDuplicates()

# 5. CREAR LA COLUMNA IMPORTE:
# Spark hace esta operación matemática de forma distribuida. Multiplica fila por fila
# la columna 'cantidad' por 'precio_unitario' y guarda el resultado en una nueva columna llamada 'importe'.
df_pedidos = df_pedidos.withColumn("importe", F.col("cantidad") * F.col("precio_unitario"))

# ==============================================================================
# Verificamos los resultados tras la limpieza y transformación
total_final_pedidos = df_pedidos.count()
print(f"\nTotal de registros DESPUÉS de limpiar: {total_final_pedidos}")
print(f"Registros eliminados (nulos críticos o duplicados): {total_inicial_pedidos - total_final_pedidos}")

# Mostramos el esquema para verificar que los tipos (integer, date, double) están correctos
# y que la columna 'importe' se ha creado con éxito.
print("\nEsquema final validado:")
df_pedidos.printSchema()

print("Datos limpios y transformados:")
df_pedidos.show(15)

Total de registros iniciales en pedidos: 120
Número de filas duplicadas antes de limpiar: 0
Recuento de nulos por columna ANTES de limpiar:
+---------+----------+-----+--------+--------+---------------+
|id_pedido|id_cliente|fecha|producto|cantidad|precio_unitario|
+---------+----------+-----+--------+--------+---------------+
|        0|         0|    0|       2|       3|              0|
+---------+----------+-----+--------+--------+---------------+


Total de registros DESPUÉS de limpiar: 120
Registros eliminados (nulos críticos o duplicados): 0

Esquema final validado:
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = false)
 |-- cantidad: double (nullable = false)
 |-- precio_unitario: double (nullable = false)
 |-- importe: double (nullable = false)

Datos limpios y transformados:
+---------+----------+----------+-----------+--------+---------------+-------+
|id_pedido|id_c

## 4. Haz un join entre clientes y pedidos.

In [29]:
# --- ANÁLISIS DE INTEGRIDAD (PRE-JOIN) ---
# Igual que hicimos en la limpieza, antes de cruzar tablas es vital saber 
# cuántos registros tenemos en origen para medir el impacto del cruce.
count_clientes = df_clientes.count()
count_pedidos = df_pedidos.count()
print(f"Registros antes del cruce: {count_clientes} Clientes y {count_pedidos} Pedidos.")

# 1. REALIZAR EL CRUCE PRINCIPAL (INNER JOIN):
# Un 'inner join' es como buscar coincidencias exactas. Spark mira la columna 
# 'id_cliente' en ambas tablas. Si el ID existe en las dos, une los datos en una sola fila.
# OJO: Si un ID está en clientes pero no en pedidos (o viceversa), esa fila SE PIERDE.
# El resultado se guarda en un nuevo DataFrame llamado 'df_ventas'.
df_ventas = df_clientes.join(df_pedidos, on="id_cliente", how="inner")

# 2. ANÁLISIS DE REGISTROS PERDIDOS (Requisito del proyecto):
# Como sabemos que el 'inner join' elimina los que no coinciden perfectamente, 
# vamos a usar una técnica analítica llamada 'Anti Join' (left_anti).
# Un left_anti join devuelve SOLO las filas de la tabla de la izquierda que NO 
# encuentran a su "pareja" en la tabla de la derecha.

# A. Encontrar pedidos "huérfanos":
# Ponemos df_pedidos a la izquierda. Si un pedido tiene un 'id_cliente' que NO 
# existe en la tabla df_clientes, el Anti Join lo captura. Esto indica un error de datos.
pedidos_sin_cliente = df_pedidos.join(df_clientes, on="id_cliente", how="left_anti")

# B. Encontrar clientes "inactivos" (que no han comprado):
# Ponemos df_clientes a la izquierda. Si un cliente no tiene ningún pedido a su 
# nombre en df_pedidos, el Anti Join lo captura. Es un cliente registrado sin ventas.
clientes_sin_pedido = df_clientes.join(df_pedidos, on="id_cliente", how="left_anti")

# ==============================================================================
# RESULTADOS Y CONCLUSIONES DEL CRUCE
# Imprimimos un pequeño informe para ver exactamente qué ha pasado en el cruce.
# ==============================================================================
print("-" * 50)
print(f"RESULTADO DEL INNER JOIN: {df_ventas.count()} ventas consolidadas.")
print("REGISTROS DESCARTADOS DURANTE EL CRUCE:")
print(f" - Pedidos perdidos (El cliente no existe en la base de datos): {pedidos_sin_cliente.count()}")
print(f" - Clientes descartados (No tienen ninguna compra asociada): {clientes_sin_pedido.count()}")
print("-" * 50)

# Mostramos los pedidos que han quedado huérfanos para poder investigar el problema
if pedidos_sin_cliente.count() > 0:
    print("\nEjemplo de pedidos huérfanos (se requiere investigar estos IDs):")
    pedidos_sin_cliente.select("id_pedido", "id_cliente", "producto").show(15)

# Mostramos cómo queda la tabla final unificada lista para el análisis de negocio
print("\nVista previa del DataFrame unificado (df_ventas) listo para analizar:")
# Seleccionamos un orden de columnas lógico para que la tabla sea fácil de leer en pantalla
df_ventas.select("id_cliente", "nombre", "ciudad", "id_pedido", "producto", "importe").show(15)

Registros antes del cruce: 40 Clientes y 120 Pedidos.
--------------------------------------------------
RESULTADO DEL INNER JOIN: 110 ventas consolidadas.
REGISTROS DESCARTADOS DURANTE EL CRUCE:
 - Pedidos perdidos (El cliente no existe en la base de datos): 10
 - Clientes descartados (No tienen ninguna compra asociada): 3
--------------------------------------------------

Ejemplo de pedidos huérfanos (se requiere investigar estos IDs):
+---------+----------+-----------+
|id_pedido|id_cliente|   producto|
+---------+----------+-----------+
|     1074|        42|  Impresora|
|     1012|        48|Desconocido|
|     1107|        47|    Monitor|
|     1114|        46|      Ratón|
|     1109|        46|Auriculares|
+---------+----------+-----------+
only showing top 5 rows

Vista previa del DataFrame unificado (df_ventas) listo para analizar:
+----------+-------+--------+---------+---------+-------+
|id_cliente| nombre|  ciudad|id_pedido| producto|importe|
+----------+-------+--------+--

## 5. Filtra ventas Premium con importe >= 100.

## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [ ]:
# TODO

## 7. Calcula agregaciones por ciudad y segmento.

In [ ]:
# TODO

## 8. Crea una vista temporal y haz una consulta SQL.

In [ ]:
# TODO

## 9. Usa `sample()` y `randomSplit()`.

In [ ]:
# TODO

## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [ ]:
# TODO

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

## 12. Cierra la sesión de Spark.

In [20]:
spark.stop()